# Reproducing the results of *"Retrieving similar past market states to time Ethereum swaps"*

This notebook regenerates **every table and figure** of the manuscript from `blocks.csv` (and, for the
language-model rows, from the shipped answer caches). It calls the same library functions as `run_all.py`.

| Section | Regenerates | Needs |
|---|---|---|
| A. Primary replay | Table 2, Figure 1 | blocks.csv (+ LLM cache or GPU for the two LLM rows) |
| B. 2-day sensitivity | Table 3 (rows 2-5) | blocks.csv |
| C. Wait-frontier audits | Table 3 (last 2 rows), Figure 3 | blocks.csv |
| D. Retrieval-signal diagnostic | Table 4 and the exploratory strata | blocks.csv |
| E. Prompt ablation | Table 5, Figure 2 | blocks.csv + ablation cache (or GPU) |
| F. Reconciliation | which numbers match the manuscript | all of the above |

**Modes** (set in the next cell): `cache` = replay the shipped LLM answers, CPU only, minutes (default);
`live` = query Ollama `qwen2.5:7b` on a GPU (about 1 h); `mock` = fake answers, software test only.
Set `SELFTEST = True` to run everything on synthetic data in about 15 s (numbers are meaningless).

In [ ]:
# ---- 0. Setup: locate the bundle, import, configure ---------------------------------------------
import os, sys, json, hashlib, time
if os.path.exists("/content") and not os.path.exists("swaprepro"):          # Google Colab
    from google.colab import drive
    drive.mount("/content/drive")
    BUNDLE = "/content/drive/MyDrive/reproducibility_package"                 # <- folder you uploaded; edit if different
    os.chdir(BUNDLE)
sys.path.insert(0, os.getcwd())
import numpy as np, pandas as pd
from IPython.display import display, Image, Markdown
from swaprepro import data, analyses as A, report as R
from swaprepro.llm import LLM

CFG = dict(
    MODE     = "cache",                 # "cache" | "live" | "mock"
    CSV      = "data/blocks.csv",
    CACHE    = "cache",
    OUT      = "results",
    SELFTEST = os.environ.get("REPRO_SELFTEST") == "1",     # synthetic data + mock LLM
    CAL_TX_PER_WIN = int(os.environ.get("CAL_TX_PER_WIN", 8)),   # swaps per CAL window in the prompt ablation
)
if CFG["SELFTEST"]:
    CFG.update(MODE="mock", OUT="results_selftest", CACHE=os.path.join("results_selftest", "cache"))
    print("!! SELFTEST: synthetic data and mock LLM; numbers are meaningless !!")
os.makedirs(CFG["OUT"], exist_ok=True); os.makedirs(CFG["CACHE"], exist_ok=True)
print({k: v for k, v in CFG.items()})

## 1. Data
Loads `blocks.csv`, checks contiguity, prints the SHA-256 (cite it in the paper), and builds the three market views used below.

In [ ]:
# Optional: create blocks.csv with BigQuery (only if data/blocks.csv does not exist yet; needs your own GCP project).
MAKE_WITH_BIGQUERY, GCP_PROJECT = False, "YOUR_GCP_PROJECT_ID"
if MAKE_WITH_BIGQUERY and not os.path.exists(CFG["CSV"]):
    from google.colab import auth; auth.authenticate_user()
    from google.cloud import bigquery
    os.makedirs("data", exist_ok=True)
    bigquery.Client(project=GCP_PROJECT).query(data.BQ_QUERY).to_dataframe().to_csv(CFG["CSV"], index=False)

if CFG["SELFTEST"]:
    b = data.synthetic_base_fee(); info = dict(rows=len(b), synthetic=True)
    CSV_USED = os.path.join(CFG["OUT"], "synthetic_blocks.csv")
    pd.DataFrame({"number": np.arange(len(b)), "base_fee_per_gas": b * 1e9}).to_csv(CSV_USED, index=False)
else:
    CSV_USED = CFG["CSV"]
    b, info = data.load_blocks(CSV_USED)
    info["sha256"] = hashlib.sha256(open(CSV_USED, "rb").read()).hexdigest()
print(info)
M1  = data.Market(b, W=7200,  feature_mode="roll")      # primary: 24-hour windows
M2  = data.Market(b, W=14400, feature_mode="roll")      # sensitivity + frontier: 48-hour windows
M2c = data.Market(b, W=14400, feature_mode="causal")    # retrieval-signal diagnostic (padded lags)
for m in (M1, M2, M2c): print(m.summary())

## A. Primary replay (Table 2, Figure 1)
16 one-day test windows; 100 swaps per window for the rule-based arms, 6 per window (96) for the language-model arms.
`kNN - heuristic` and its CI are **new** outputs (they were not printed by the original notebook).

In [ ]:
llm = LLM(CFG["MODE"], cache_path=os.path.join(CFG["CACHE"], "ollama_qwen2.5_7b_test_cache.json"))
primary = A.run_primary(M1, llm)
t2 = R.table2(primary); t2.to_csv(f"{CFG['OUT']}/Table2_primary.csv", index=False)
R.fig1(primary, f"{CFG['OUT']}/Figure1.png")
display(t2)
print("Matched-wait paired comparisons (language model with retrieval):")
display(pd.DataFrame(primary["matched"]))
print("kNN minus heuristic (unequal waits):", {k: round(v, 3) for k, v in primary["knn_vs_heur"].items()})
print("Collapse guard (wait in blocks -> number of swaps):", primary["collapse"])
display(Image(f"{CFG['OUT']}/Figure1.png", width=700))

## B. Two-day sensitivity harness (Table 3)
Static library vs. rolling 5,000-block look-back; then the two-sided matched-wait step.
The harness summarises neighbours with the **median** (as in the original 2-day runs); the primary run uses the **mean**.

In [ ]:
h_static = A.run_harness(M2, "static",  matched=True)
h_roll   = A.run_harness(M2, "rolling", matched=True)
for h in (h_static, h_roll):
    print(f"\n=== {h['retriever']} retriever | mean freshness lag {h['freshness_lag']:.0f} blocks ===")
    display(h["table"].round(2))
    print("retrieval - heuristic:", {k: round(v, 2) for k, v in h["delta"].items()})
print("\nRolling retriever, matched-wait comparisons:"); display(pd.DataFrame(h_roll["matched"]).T)

## C. Wait-frontier audits (Table 3, Figure 3)
Two audits: the first (margin +/-1.0 pp) and the later one (margin +/-0.40 pp, Pareto envelope). Both are reported in the paper.

In [ ]:
fr_init, fr_hard = A.frontier_audit(M2, "initial"), A.frontier_audit(M2, "hardened")
for fr in (fr_init, fr_hard):
    print(f"{fr['variant']:9s} margin +/-{fr['margin']:.2f} pp | mean gap {fr['mean_gap']:+.2f} "
          f"[{fr['ci_lo']:+.2f}, {fr['ci_hi']:+.2f}] | {fr['verdict']} | sign changes near {[round(x) for x in fr['crossings_s']]} s")
R.fig3(fr_hard, f"{CFG['OUT']}/Figure3_frontier_hardened.png"); R.fig3(fr_init, f"{CFG['OUT']}/Figure3_frontier_initial.png")
t3 = R.table3(h_static, h_roll, fr_init, fr_hard, primary); t3.to_csv(f"{CFG['OUT']}/Table3_sensitivity.csv", index=False)
display(t3); display(Image(f"{CFG['OUT']}/Figure3_frontier_hardened.png", width=800))

## D. Retrieval-signal diagnostic (Table 4)
Incremental correlation between the kNN prediction and the realised best saving after removing the trailing-rank decile mean
(both sides), bootstrapped over the six calibration windows; BH-FDR over terciles and over 12 exploratory strata.

In [ ]:
diag = A.diagnostic(M2c)
t4 = R.table4(diag); t4.to_csv(f"{CFG['OUT']}/Table4_diagnostic.csv", index=False)
diag["strata"].to_csv(f"{CFG['OUT']}/Table4_strata.csv", index=False)
display(t4); print("Exploratory strata (sorted by p; post hoc, treat as hypotheses):"); display(diag["strata"].round(4))

## E. Prompt ablation on the calibration split (Table 5, Figure 2)
Nine prompt variants x 104 swaps x 11 decision states (about 10,300 calls). In `cache` mode the shipped answers are replayed.

In [ ]:
abl = A.run_ablation(CSV_USED, CFG["CACHE"], os.path.join(CFG["OUT"], "ablation"), mode=CFG["MODE"],
                     cal_tx_per_win=CFG["CAL_TX_PER_WIN"])
t5 = R.table5(abl); t5.to_csv(f"{CFG['OUT']}/Table5_ablation.csv", index=False)
R.fig2(abl, f"{CFG['OUT']}/Figure2.png")
display(t5); display(Image(f"{CFG['OUT']}/Figure2.png", width=800))

## F. Reconciliation with the manuscript
Every number printed in the manuscript's Results has a row in `expected_results.json`. `OK` = regenerated value is within the stated
tolerance; `DIFFERS` = update the manuscript (or investigate); `not run` = the section needed was skipped.
With the same `blocks.csv` and cache files all rows should read `OK`.

In [ ]:
flat = R.flatten(primary, h_static, h_roll, fr_init, fr_hard, diag, abl)
rec = R.reconcile(flat, "expected_results.json"); rec.to_csv(f"{CFG['OUT']}/reconciliation.csv", index=False)
print(rec.status.value_counts().to_string())
display(rec[rec.status != "OK"] if (rec.status != "OK").any() else Markdown("**All rows match the manuscript.**"))
results = dict(meta=dict(info=info, mode=CFG["MODE"], selftest=CFG["SELFTEST"]), primary=primary, harness_static=h_static,
               harness_rolling=h_roll, frontier_initial=fr_init, frontier_hardened=fr_hard, diagnostic=diag, ablation=abl)
json.dump(R.jsonable(results), open(f"{CFG['OUT']}/results.json", "w"), indent=1)
print("Saved tables, figures, reconciliation.csv and results.json in", CFG["OUT"])

### Notes
* **Determinism.** Rule-based results are deterministic given `blocks.csv` (all random draws are seeded). The LLM rows are replayed from cache, so they are identical; regenerating them with `mode="live"` can differ slightly if the Ollama build or GPU differs (the ablation records a determinism re-query: 100% agreement in our run).
* **Windows.** Two window lengths are used on purpose (see manuscript, Analysis history). Section A uses 24-hour windows; B, C and D use 48-hour windows.
* **What is not reproduced here:** access to BigQuery (needs your project) and the original Ollama binary/GPU.
* **Citing.** Report the printed SHA-256 of `blocks.csv`, the model digest (`845dbda0ea48`) and the package version in the paper.